# 7.5 实用工具库

本 notebook 介绍几个在 PyTorch 项目中常用的工具库：TorchMetrics、Albumentations 和集成学习方法

### 学习目标
- 掌握 TorchMetrics 的三步工作流
- 了解 Albumentations 的数据增强接口
- 学会模型集成的多种实现方式

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print(f"PyTorch 版本: {torch.__version__}")

## 1. TorchMetrics

TorchMetrics 是 PyTorch 生态中的标准化指标库，由 Lightning AI 开发维护

### 安装
```bash
pip install torchmetrics
```

### 为什么使用 TorchMetrics？
- 自动处理分布式环境下的指标同步
- 支持累积计算（不是简单的批次平均）
- 提供 100+ 种预定义指标
- 与 PyTorch 无缝集成

In [ ]:
# 检查 torchmetrics 是否安装
try:
    import torchmetrics
    print(f"TorchMetrics 版本: {torchmetrics.__version__}")
    HAS_TORCHMETRICS = True
except ImportError:
    print("TorchMetrics 未安装")
    print("安装命令: pip install torchmetrics")
    HAS_TORCHMETRICS = False

### 1.1 三步工作流：创建 -> 更新 -> 计算

所有 TorchMetrics 指标都遵循相同的模式：

```python
# 步骤 1: 创建指标
metric = torchmetrics.Accuracy(task="multiclass", num_classes=5)

# 步骤 2: 每个 batch 更新
metric.update(preds, target)

# 步骤 3: 计算最终结果
result = metric.compute()

# 重置（新 epoch 开始时）
metric.reset()
```

In [ ]:
if HAS_TORCHMETRICS:
    from torchmetrics import Accuracy, F1Score, Precision, Recall

    # 创建指标
    num_classes = 5
    accuracy = Accuracy(task="multiclass", num_classes=num_classes)
    f1 = F1Score(task="multiclass", num_classes=num_classes, average="macro")
    precision = Precision(task="multiclass", num_classes=num_classes, average="macro")
    recall = Recall(task="multiclass", num_classes=num_classes, average="macro")

    # 模拟多个 batch 的预测结果
    torch.manual_seed(42)
    for batch_idx in range(5):
        # 模拟预测概率和真实标签
        preds = torch.randn(32, num_classes)  # 未归一化的 logits
        target = torch.randint(0, num_classes, (32,))

        # 更新指标
        accuracy.update(preds, target)
        f1.update(preds, target)
        precision.update(preds, target)
        recall.update(preds, target)

    # 计算最终结果
    print("=== TorchMetrics 结果（累积所有 batch）===")
    print(f"Accuracy:  {accuracy.compute():.4f}")
    print(f"F1 Score:  {f1.compute():.4f}")
    print(f"Precision: {precision.compute():.4f}")
    print(f"Recall:    {recall.compute():.4f}")

    # 重置
    accuracy.reset()
    f1.reset()
    print("\n指标已重置")
else:
    print("跳过 TorchMetrics 演示（未安装）")
    print("\n三步工作流:")
    print("  1. metric = Accuracy(task='multiclass', num_classes=5)")
    print("  2. metric.update(preds, target)  # 每个 batch")
    print("  3. result = metric.compute()      # epoch 结束时")

### 1.2 MetricCollection：批量管理指标

In [ ]:
if HAS_TORCHMETRICS:
    from torchmetrics import MetricCollection

    # 使用 MetricCollection 一次管理多个指标
    metrics = MetricCollection({
        'accuracy': Accuracy(task="multiclass", num_classes=5),
        'f1': F1Score(task="multiclass", num_classes=5, average="macro"),
        'precision': Precision(task="multiclass", num_classes=5, average="macro"),
        'recall': Recall(task="multiclass", num_classes=5, average="macro"),
    })

    # 模拟训练
    torch.manual_seed(42)
    for batch_idx in range(5):
        preds = torch.randn(32, 5)
        target = torch.randint(0, 5, (32,))
        # 一次性更新所有指标
        metrics.update(preds, target)

    # 一次性计算所有指标
    results = metrics.compute()
    print("=== MetricCollection 结果 ===")
    for name, value in results.items():
        print(f"  {name}: {value:.4f}")

    metrics.reset()
else:
    print("跳过 MetricCollection 演示")

### 1.3 自定义指标

In [ ]:
if HAS_TORCHMETRICS:
    from torchmetrics import Metric

    class MeanAbsolutePercentageError(Metric):
        """平均绝对百分比误差 (MAPE)

        MAPE = (1/n) * sum(|y_true - y_pred| / |y_true|) * 100
        """

        def __init__(self):
            super().__init__()
            # 定义需要累积的状态
            self.add_state("sum_ape", default=torch.tensor(0.0), dist_reduce_fx="sum")
            self.add_state("count", default=torch.tensor(0), dist_reduce_fx="sum")

        def update(self, preds: torch.Tensor, target: torch.Tensor) -> None:
            """更新状态"""
            # 避免除以零
            mask = target != 0
            ape = torch.abs((target[mask] - preds[mask]) / target[mask])
            self.sum_ape += ape.sum()
            self.count += mask.sum()

        def compute(self) -> torch.Tensor:
            """计算最终结果"""
            return (self.sum_ape / self.count) * 100

    # 测试自定义指标
    mape = MeanAbsolutePercentageError()
    preds = torch.tensor([105.0, 200.0, 310.0])
    target = torch.tensor([100.0, 210.0, 300.0])

    mape.update(preds, target)
    result = mape.compute()
    print(f"MAPE: {result:.2f}%")
    print("\n自定义指标关键点:")
    print("  1. 继承 torchmetrics.Metric")
    print("  2. __init__ 中用 add_state 定义状态")
    print("  3. update 中更新状态")
    print("  4. compute 中计算最终结果")
else:
    print("跳过自定义指标演示")

### 1.4 在训练循环中使用 TorchMetrics

In [ ]:
def train_with_torchmetrics():
    """使用 TorchMetrics 的完整训练示例"""

    # 准备数据
    torch.manual_seed(42)
    num_classes = 5
    X_train = torch.randn(200, 20)
    y_train = (X_train[:, :num_classes].argmax(dim=1)).long()
    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=32,
        shuffle=True,
    )

    # 模型
    model = nn.Sequential(
        nn.Linear(20, 64),
        nn.ReLU(),
        nn.Linear(64, num_classes),
    )
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    if HAS_TORCHMETRICS:
        # 使用 TorchMetrics
        train_metrics = MetricCollection({
            'acc': Accuracy(task="multiclass", num_classes=num_classes),
            'f1': F1Score(task="multiclass", num_classes=num_classes, average="macro"),
        })

        for epoch in range(5):
            model.train()
            train_metrics.reset()  # 每个 epoch 开始时重置

            epoch_loss = 0.0
            for data, target in train_loader:
                output = model(data)
                loss = criterion(output, target)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()
                train_metrics.update(output, target)  # 更新指标

            # epoch 结束时计算指标
            results = train_metrics.compute()
            avg_loss = epoch_loss / len(train_loader)
            print(
                f"Epoch {epoch+1} | "
                f"Loss: {avg_loss:.4f} | "
                f"Acc: {results['acc']:.4f} | "
                f"F1: {results['f1']:.4f}"
            )
    else:
        # 不使用 TorchMetrics 的简化版
        for epoch in range(5):
            model.train()
            correct = 0
            total = 0
            epoch_loss = 0.0

            for data, target in train_loader:
                output = model(data)
                loss = criterion(output, target)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()
                pred = output.argmax(dim=1)
                correct += (pred == target).sum().item()
                total += target.size(0)

            avg_loss = epoch_loss / len(train_loader)
            acc = correct / total
            print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Acc: {acc:.4f}")


train_with_torchmetrics()

## 2. Albumentations

Albumentations 是一个高性能的图像增强库，特别适合计算机视觉任务

### 安装
```bash
pip install albumentations
```

### 特点
- 速度比 torchvision.transforms 更快
- 支持分割、检测等任务的同步变换（图像和标签同时变换）
- 提供更丰富的增强方法

In [ ]:
# 检查 albumentations 是否安装
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    import numpy as np
    print(f"Albumentations 版本: {A.__version__}")
    HAS_ALBUMENTATION = True
except ImportError:
    print("Albumentations 未安装")
    print("安装命令: pip install albumentations")
    HAS_ALBUMENTATION = False

### 2.1 Compose API

使用 `A.Compose` 将多个变换组合在一起

### 变换类型
- **ImageOnlyTransform**: 只变换图像（亮度、对比度等）
- **DualTransform**: 同时变换图像和标签（翻转、旋转等）

In [ ]:
if HAS_ALBUMENTATION:
    # 定义训练增强（分类任务）
    train_transform = A.Compose([
        # DualTransform: 图像和标签同步变换
        A.Resize(224, 224),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.1),
        A.RandomRotate90(p=0.3),

        # ImageOnlyTransform: 只变换图像
        A.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.1,
            p=0.5,
        ),
        A.GaussNoise(p=0.3),

        # 归一化 + 转为 Tensor
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
        ToTensorV2(),
    ])

    # 验证增强（不做随机变换）
    val_transform = A.Compose([
        A.Resize(224, 224),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
        ToTensorV2(),
    ])

    # 测试
    # Albumentations 使用 numpy 数组（HWC 格式）
    dummy_image = np.random.randint(0, 255, (300, 400, 3), dtype=np.uint8)
    print(f"原始图像: shape={dummy_image.shape}, dtype={dummy_image.dtype}")

    # 应用变换
    result = train_transform(image=dummy_image)
    transformed_image = result['image']
    print(f"增强后: shape={transformed_image.shape}, dtype={transformed_image.dtype}")
    print(f"值范围: [{transformed_image.min():.2f}, {transformed_image.max():.2f}]")
else:
    print("Albumentations Compose API 示例:")
    print()
    print("train_transform = A.Compose([")
    print("    A.Resize(224, 224),            # DualTransform")
    print("    A.HorizontalFlip(p=0.5),       # DualTransform")
    print("    A.ColorJitter(..., p=0.5),      # ImageOnlyTransform")
    print("    A.Normalize(mean=..., std=...), # ImageOnlyTransform")
    print("    ToTensorV2(),                   # numpy -> tensor")
    print("])")
    print()
    print("# 使用: result = train_transform(image=image)")
    print("# 获取: transformed_image = result['image']")

### 2.2 在 Dataset 中使用（分割任务）

在语义分割等任务中，增强需要同时作用于图像和掩码

In [ ]:
if HAS_ALBUMENTATION:
    from torch.utils.data import Dataset

    class SegmentationDataset(Dataset):
        """语义分割数据集（使用 Albumentations）"""

        def __init__(self, images, masks, transform=None):
            self.images = images
            self.masks = masks
            self.transform = transform

        def __len__(self):
            return len(self.images)

        def __getitem__(self, idx):
            image = self.images[idx]  # numpy array, HWC
            mask = self.masks[idx]    # numpy array, HW

            if self.transform:
                # 关键: image 和 mask 一起传入，确保同步变换
                result = self.transform(image=image, mask=mask)
                image = result['image']
                mask = result['mask']

            return image, mask

    # 分割任务的增强
    seg_transform = A.Compose([
        A.Resize(256, 256),
        A.HorizontalFlip(p=0.5),        # 图像和掩码同时翻转
        A.RandomRotate90(p=0.3),         # 图像和掩码同时旋转
        A.ColorJitter(brightness=0.2, p=0.3),  # 只改变图像颜色
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

    # 创建合成数据
    dummy_images = [np.random.randint(0, 255, (300, 400, 3), dtype=np.uint8) for _ in range(10)]
    dummy_masks = [np.random.randint(0, 3, (300, 400), dtype=np.uint8) for _ in range(10)]

    dataset = SegmentationDataset(dummy_images, dummy_masks, transform=seg_transform)
    image, mask = dataset[0]
    print(f"增强后的图像: shape={image.shape}, dtype={image.dtype}")
    print(f"增强后的掩码: shape={mask.shape}, dtype={mask.dtype}")
    print(f"掩码类别: {mask.unique().tolist()}")
    print("\n关键点: DualTransform（翻转、旋转）会同步作用于 image 和 mask")
else:
    print("分割任务 Dataset 中使用 Albumentations:")
    print()
    print("def __getitem__(self, idx):")
    print("    image = self.images[idx]")
    print("    mask = self.masks[idx]")
    print("    # 同时传入 image 和 mask")
    print("    result = self.transform(image=image, mask=mask)")
    print("    return result['image'], result['mask']")

## 3. 模型集成

模型集成通过组合多个模型的预测来提升性能。常见方法：

| 方法 | 思路 | 特点 |
|------|------|------|
| Voting | 多个模型投票 | 简单有效 |
| Bagging | 训练多个模型取平均 | 减少方差 |
| Fusion | 加权融合模型输出 | 更灵活 |

### 3.1 TorchEnsemble（可选库）

```bash
pip install torchensemble
```

TorchEnsemble 提供了标准化的集成方法：
- `VotingClassifier`: 投票分类
- `BaggingClassifier`: Bagging 集成
- `FusionClassifier`: 特征融合

```python
from torchensemble import VotingClassifier

# 基础模型
model = VotingClassifier(
    estimator=MyModel,
    n_estimators=5,
    cuda=True,
)
model.fit(train_loader, epochs=10, test_loader=val_loader)
```

### 3.2 手动集成：ModuleList 方式

使用 `nn.ModuleList` 管理多个子模型，将它们的输出进行融合

In [ ]:
class EnsembleModel(nn.Module):
    """使用 ModuleList 的集成模型

    将多个子模型的输出取平均
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        output_dim: int,
        n_models: int = 3,
    ):
        super().__init__()
        self.models = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(hidden_dim, output_dim),
            )
            for _ in range(n_models)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """前向传播：对所有子模型输出取平均"""
        outputs = [model(x) for model in self.models]
        # 堆叠后取平均: (n_models, batch, classes) -> (batch, classes)
        return torch.stack(outputs, dim=0).mean(dim=0)


# 创建集成模型
ensemble = EnsembleModel(
    input_dim=20,
    hidden_dim=32,
    output_dim=5,
    n_models=3,
)

print(f"集成模型包含 {len(ensemble.models)} 个子模型")
total_params = sum(p.numel() for p in ensemble.parameters())
print(f"总参数量: {total_params:,}")

# 测试
x = torch.randn(8, 20)
output = ensemble(x)
print(f"\n输入: {x.shape}")
print(f"输出: {output.shape}")

### 3.3 手动集成：投票方式

训练多个独立模型，推理时对预测结果进行投票

In [ ]:
class VotingEnsemble:
    """投票集成

    管理多个独立训练的模型，推理时进行投票或概率平均
    """

    def __init__(self, models: list, mode: str = "soft"):
        """
        Args:
            models: 模型列表
            mode: 'hard'（多数投票）或 'soft'（概率平均）
        """
        self.models = models
        self.mode = mode

    @torch.no_grad()
    def predict(self, x: torch.Tensor) -> torch.Tensor:
        """集成预测

        Args:
            x: 输入张量

        Returns:
            预测类别
        """
        for model in self.models:
            model.eval()

        if self.mode == "soft":
            # 软投票：对概率取平均
            probs = []
            for model in self.models:
                output = model(x)
                prob = torch.softmax(output, dim=1)
                probs.append(prob)
            avg_probs = torch.stack(probs, dim=0).mean(dim=0)
            return avg_probs.argmax(dim=1)

        else:
            # 硬投票：对预测类别投票
            votes = []
            for model in self.models:
                output = model(x)
                pred = output.argmax(dim=1)
                votes.append(pred)
            # 堆叠后取众数
            votes = torch.stack(votes, dim=0)  # (n_models, batch)
            # 对每个样本找出现次数最多的类别
            result = torch.mode(votes, dim=0).values
            return result


# 创建多个独立模型
models = []
for i in range(3):
    torch.manual_seed(i)  # 不同的初始化
    m = nn.Sequential(
        nn.Linear(20, 32),
        nn.ReLU(),
        nn.Linear(32, 5),
    )
    models.append(m)

# 创建投票集成
voter_soft = VotingEnsemble(models, mode="soft")
voter_hard = VotingEnsemble(models, mode="hard")

# 测试
x_test = torch.randn(16, 20)
pred_soft = voter_soft.predict(x_test)
pred_hard = voter_hard.predict(x_test)

print(f"软投票预测: {pred_soft.tolist()}")
print(f"硬投票预测: {pred_hard.tolist()}")
print(f"两种方式一致: {(pred_soft == pred_hard).float().mean():.2f}")

In [ ]:
# 完整示例：训练集成模型并对比
def train_and_compare():
    """训练单模型和集成模型，对比性能"""
    # 数据
    torch.manual_seed(42)
    num_classes = 5
    X = torch.randn(300, 20)
    y = (X[:, :num_classes].argmax(dim=1)).long()

    X_train, X_test = X[:200], X[200:]
    y_train, y_test = y[:200], y[200:]

    criterion = nn.CrossEntropyLoss()

    # 训练 3 个独立模型
    trained_models = []
    for i in range(3):
        torch.manual_seed(i * 10)
        model = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )
        optimizer = optim.Adam(model.parameters(), lr=0.01)

        model.train()
        for epoch in range(30):
            output = model(X_train)
            loss = criterion(output, y_train)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        trained_models.append(model)

    # 评估各个模型
    print("=== 各模型单独表现 ===")
    for i, model in enumerate(trained_models):
        model.eval()
        with torch.no_grad():
            pred = model(X_test).argmax(dim=1)
            acc = (pred == y_test).float().mean()
        print(f"模型 {i+1}: 准确率 = {acc:.4f}")

    # 集成评估
    print("\n=== 集成模型表现 ===")
    voter = VotingEnsemble(trained_models, mode="soft")
    ensemble_pred = voter.predict(X_test)
    ensemble_acc = (ensemble_pred == y_test).float().mean()
    print(f"软投票集成: 准确率 = {ensemble_acc:.4f}")

    voter_hard = VotingEnsemble(trained_models, mode="hard")
    hard_pred = voter_hard.predict(X_test)
    hard_acc = (hard_pred == y_test).float().mean()
    print(f"硬投票集成: 准确率 = {hard_acc:.4f}")


train_and_compare()

## 4. 总结

### TorchMetrics
- **三步工作流**: 创建 -> update() -> compute()
- **MetricCollection**: 批量管理多个指标
- **自定义指标**: 继承 `Metric`，实现 `update` 和 `compute`
- **优势**: 自动处理分布式同步，正确的累积计算

### Albumentations
- **Compose**: 组合多个变换
- **DualTransform**: 同时变换图像和标签（翻转、旋转等）
- **ImageOnlyTransform**: 只变换图像（颜色、噪声等）
- **优势**: 速度快，支持分割/检测任务的同步变换

### 模型集成

| 方式 | 实现 | 适用场景 |
|------|------|----------|
| ModuleList | 端到端训练 | 需要一起训练的集成 |
| 软投票 | 概率平均 | 独立训练的模型 |
| 硬投票 | 类别投票 | 简单快速 |
| TorchEnsemble | 标准化接口 | 快速实验 |

---

## 练习题

### 基础题
1. 使用 TorchMetrics（或手动计算）在一个训练循环中跟踪 Accuracy、Precision、Recall、F1
2. 列出 3 个 Albumentations 的 DualTransform 和 3 个 ImageOnlyTransform

### 进阶题
3. 使用 TorchMetrics 实现一个自定义指标：Top-K 准确率
   - 如果真实标签在预测概率最高的 K 个类别中，就算正确
   - 支持 K 作为参数

### 挑战题
4. 实现一个加权集成模型：
   - 训练 N 个子模型
   - 在验证集上评估每个子模型的准确率
   - 根据准确率自动分配投票权重
   - 对比均匀权重和自适应权重的集成效果

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的更多练习题来巩固知识。